In [ ]:
import numpy as np
import pandas as pd
import os
!pip install wandb -q

import wandb
import os
!pip install pytorch-forecasting pandas numpy torch matplotlib


In [ ]:
%run data_extraction_feature_engineering.ipynb
df_train_full, df_test_full = extract_data()

Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  .

In [ ]:
import pandas as pd
import numpy as np
from pytorch_forecasting.data import TimeSeriesDataSet
from pytorch_forecasting.models.nbeats import NBeats

df = df_train_full.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

unique_dates = df['Date'].sort_values().unique()
date_to_time_idx = {date: idx for idx, date in enumerate(unique_dates)}
df['time_idx'] = df['Date'].map(date_to_time_idx)
df['store_id'] = df['Store'].astype(str)
df['dept_id'] = df['Dept'].astype(str)
df['type_cat'] = df['Type'].astype(str)
df['Weekly_Sales'] = df['Weekly_Sales'].fillna(0)
external_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
for col in external_cols:
    if col in df.columns:
        df[col] = df[col].interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')

VAL_START_DATE = pd.Timestamp('2012-09-01')
max_time_idx = df[df['Date'] < VAL_START_DATE]['time_idx'].max()

MAX_ENCODER_LENGTH = 26
MIN_PREDICT_LENGTH = 8
MAX_PREDICT_LENGTH = 8

training_dataset = TimeSeriesDataSet(
    data=df[df['time_idx'] <= max_time_idx],
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["store_id", "dept_id"],

    time_varying_unknown_reals=["Weekly_Sales"],

    min_encoder_length=MAX_ENCODER_LENGTH,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=MAX_PREDICT_LENGTH,
    max_prediction_length=MAX_PREDICT_LENGTH,

    allow_missing_timesteps=True
)

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data=df,
    min_prediction_idx=max_time_idx + 1,
    max_prediction_length=MAX_PREDICT_LENGTH
)

print("Datasets created successfully!")

print(f"Training samples: {len(training_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")


/tmp/ipykernel_77564/1309483596.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
/home/sandro/anaconda3/envs/cs224n/lib/python3.13/site-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 214 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_id': '1', '__group_id__dept_id': '77'}, {'__group_id__store_id': '1', '__group_id__dept_id': '78'}, {'__group_id__store_id': '10', '__group_id__dept_id': '77'}, {'__group_id__store_id': '10', '__group_id__dept_id': '78'}, {'__group_id__store_id': '11', '__group_id__dept_id': '48'}, {'__group_id__store_id': '11', '__grou

Datasets created successfully!
Training samples: 293284
Validation samples: 2909


/home/sandro/anaconda3/envs/cs224n/lib/python3.13/site-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 274 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_id': '1', '__group_id__dept_id': '45'}, {'__group_id__store_id': '1', '__group_id__dept_id': '47'}, {'__group_id__store_id': '1', '__group_id__dept_id': '99'}, {'__group_id__store_id': '10', '__group_id__dept_id': '45'}, {'__group_id__store_id': '10', '__group_id__dept_id': '47'}, {'__group_id__store_id': '10', '__group_id__dept_id': '94'}, {'__group_id__store_id': '10', '__group_id__dept_id': '98'}, {'__group_id__store_id': '11', '__group_id__dept_id': '45'}, {'__group_id__store_id': '11', '__group_id__dept_id': '47'}, {'__group_id__store_id': '11', '__group_id__dept_id': '51'}]
 

In [ ]:
batch_size = 32

train_dataloader = training_dataset.to_dataloader(
    batch_size=batch_size,
    num_workers=0,
    shuffle=True
)

val_dataloader = validation_dataset.to_dataloader(
    batch_size=batch_size,
    num_workers=0,
    shuffle=False
)

print("Dataloaders created!")
print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")


Dataloaders created!
Train batches: 9165
Val batches: 90


In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from pytorch_forecasting.models.nbeats import NBeats

wandb_logger = WandbLogger(
    project="walmart-sales-forecasting",
    job_type="NBEATS_Training",
    name="NBEATS_Val_Evaluation",
    config={
        "max_encoder_length": MAX_ENCODER_LENGTH,
        "max_prediction_length": MAX_PREDICT_LENGTH,
        "hidden_size": 8,
        "learning_rate": 0.001
    }
)

model = NBeats.from_dataset(
    training_dataset,
    learning_rate=0.001,
    widths=[32],
    num_blocks=[3],
    num_block_layers=[2],
    stack_types=['generic'],
    log_interval=10
)





/home/sandro/anaconda3/envs/cs224n/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/sandro/anaconda3/envs/cs224n/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from pytorch_forecasting.models.nbeats import NBeats

wandb_logger = WandbLogger(
    project="walmart-sales-forecasting_NBeats",
    job_type="NBEATS_Training",
    name="NBEATS_Univariate_Val_Evaluation",
    config={
        "max_encoder_length": MAX_ENCODER_LENGTH,
        "max_prediction_length": MAX_PREDICT_LENGTH,
        "widths": [32],
        "num_blocks": [3],
        "num_block_layers": [2],
        "stack_types": ["generic"],
        "learning_rate": 0.001
    }
)

model = NBeats.from_dataset(
    training_dataset,
    learning_rate=0.001,
    widths=[32],
    num_blocks=[3],
    num_block_layers=[2],
    stack_types=['generic'],
    log_interval=10
)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {num_params:,}")

early_stop_callback = EarlyStopping(monitor="val_loss", patience=3, mode="min")
lr_monitor = LearningRateMonitor(logging_interval='step')

trainer = pl.Trainer(
    max_epochs=5,
    devices=1,
    callbacks=[early_stop_callback, lr_monitor],
    logger=wandb_logger,
    enable_checkpointing=False,
    gradient_clip_val=0.1
)

trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

print("Training Complete!")


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/sandro/.netrc.


Number of trainable parameters: 6,456


wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss            │ MASE       │      0 │ train │     0 │
│ 1 │ logging_metrics │ ModuleList │      0 │ train │     0 │
│ 2 │ net_blocks      │ ModuleList │  6.5 K │ train │     0 │
└───┴─────────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 6.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.5 K                                                                                                
Total estimated model params size (MB): 0.026                                                                      
Modules in train mode: 41                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()